# Predicting `is_fraud` from `shop.db`

This notebook builds a complete CRISP-DM machine learning pipeline to predict whether an order is fraudulent using the `orders` table in `shop.db`.

## CRISP-DM Sections
1. Business Understanding
2. Data Understanding
3. Data Preparation
4. Modeling
5. Evaluation
6. Deployment

In [2]:
import sqlite3
import pandas as pd
import numpy as np

## 1. Business Understanding
The goal of this project is to build a machine learning model that predicts whether an order is fraudulent (`is_fraud`) so the business can identify risky orders and support fraud review.

This model could be used to flag high-risk orders for manual review or automated fraud prevention systems.

Success will not be measured only by overall accuracy, since the dataset is imbalanced. Instead, correctly identifying fraudulent transactions (recall for the fraud class) is especially important.

## 2. Data Understanding
Connect to the SQLite database, inspect the available tables, review the `orders` table, and explore the target variable.

The dataset was loaded from a SQLite database and the `orders` table was explored.

Initial exploration included reviewing the structure of the data (`head`, `info`), summary statistics (`describe`), and checking for missing values.

The target variable `is_fraud` is highly imbalanced, with the majority of transactions labeled as non-fraud. This imbalance is important because it can affect model performance.

Some relationships between features and fraud were also explored. For example, fraud rates may vary across payment methods, device types, or geographic locations.

In [3]:
# Connect to the SQLite database
conn = sqlite3.connect("shop.db")

# Check available tables
tables = pd.read_sql("""
SELECT name 
FROM sqlite_master 
WHERE type='table';
""", conn)

tables

,name
0,customers
1,sqlite_sequence
2,products
3,orders
4,order_items
5,shipments
6,product_reviews


In [4]:
# Load the orders table
orders = pd.read_sql("SELECT * FROM orders", conn)

# Preview the first rows
orders.head()

,order_id,customer_id,order_datetime,billing_zip,shipping_zip,shipping_state,payment_method,device_type,ip_country,promo_used,promo_code,order_subtotal,shipping_fee,tax_amount,order_total,risk_score,is_fraud
0,1,1,2025-11-29 00:51:07,28289,28289,CO,card,mobile,US,0,None,662.95,15.44,46.30,724.69,38.3,0
1,2,1,2025-09-01 10:25:59,28289,13888,NY,card,desktop,US,1,SAVE10,862.92,14.74,66.61,944.27,94.9,0
2,3,1,2025-12-15 07:24:41,28289,28289,CO,card,mobile,US,0,None,796.09,14.04,40.72,850.85,53.8,1
3,4,1,2025-11-06 18:21:19,28289,28289,CO,bank,mobile,US,1,WELCOME,137.60,6.99,11.88,156.47,4.2,0
4,5,1,2025-11-30 05:34:15,28289,28289,CO,card,mobile,CA,0,None,17.07,6.99,1.40,25.46,4.9,0


In [5]:
# Shape and columns
print("Shape:", orders.shape)
print("\nColumns:")
print(orders.columns.tolist())

Shape: (5000, 17)

Columns:
['order_id', 'customer_id', 'order_datetime', 'billing_zip', 'shipping_zip', 'shipping_state', 'payment_method', 'device_type', 'ip_country', 'promo_used', 'promo_code', 'order_subtotal', 'shipping_fee', 'tax_amount', 'order_total', 'risk_score', 'is_fraud']


In [6]:
# Data types and missing values
orders.info()

print("\nMissing values by column:")
print(orders.isnull().sum().sort_values(ascending=False))

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 17 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   order_id        5000 non-null   int64  
 1   customer_id     5000 non-null   int64  
 2   order_datetime  5000 non-null   object 
 3   billing_zip     5000 non-null   object 
 4   shipping_zip    5000 non-null   object 
 5   shipping_state  5000 non-null   object 
 6   payment_method  5000 non-null   object 
 7   device_type     5000 non-null   object 
 8   ip_country      5000 non-null   object 
 9   promo_used      5000 non-null   int64  
 10  promo_code      1261 non-null   object 
 11  order_subtotal  5000 non-null   float64
 12  shipping_fee    5000 non-null   float64
 13  tax_amount      5000 non-null   float64
 14  order_total     5000 non-null   float64
 15  risk_score      5000 non-null   float64
 16  is_fraud        5000 non-null   int64  
dtypes: float64(5), int64(4), object(8

In [7]:
# Target distribution
print("is_fraud counts:")
print(orders["is_fraud"].value_counts(dropna=False))

print("\nis_fraud proportions:")
print(orders["is_fraud"].value_counts(normalize=True, dropna=False))

is_fraud counts:
is_fraud
0    4682
1     318
Name: count, dtype: int64

is_fraud proportions:
is_fraud
0    0.9364
1    0.0636
Name: proportion, dtype: float64


In [8]:
orders.describe()

,order_id,customer_id,promo_used,order_subtotal,shipping_fee,tax_amount,order_total,risk_score,is_fraud
count,5000.000000,5000.000000,5000.000000,5000.000000,5000.000000,5000.000000,5000.000000,5000.000000,5000.000000
mean,2500.500000,28.448200,0.252200,384.144678,9.668290,27.738312,421.551280,26.115940,0.063600
std,1443.520003,49.086939,0.434319,284.124017,5.126125,21.446470,305.183694,27.433842,0.244063
min,1.000000,1.000000,0.000000,4.730000,0.000000,0.250000,5.380000,0.100000,0.000000
25%,1250.750000,2.000000,0.000000,164.180000,6.990000,11.470000,185.760000,4.600000,0.000000
50%,2500.500000,6.000000,0.000000,330.720000,8.040000,23.365000,364.835000,14.500000,0.000000
75%,3750.250000,28.000000,1.000000,549.752500,12.990000,39.152500,596.940000,40.225000,0.000000
max,5000.000000,250.000000,1.000000,1921.170000,28.140000,148.130000,2053.110000,100.000000,1.000000


In [9]:
print("Payment method:")
print(orders["payment_method"].value_counts())

print("\nDevice type:")
print(orders["device_type"].value_counts())

print("\nShipping state:")
print(orders["shipping_state"].value_counts())

Payment method:
payment_method
card      3128
paypal    1050
bank       725
crypto      97
Name: count, dtype: int64

Device type:
device_type
mobile     2734
desktop    1902
tablet      364
Name: count, dtype: int64

Shipping state:
shipping_state
CO    1702
OH     390
MI     367
TX     350
NC     338
AZ     296
FL     244
IL     232
WA     164
NY     149
UT     148
PA     147
GA     112
OR     106
CA     100
MA      57
NJ      51
VA      47
Name: count, dtype: int64


In [10]:
orders.groupby("payment_method")["is_fraud"].mean().sort_values(ascending=False)

payment_method
crypto    0.103093
card      0.067455
bank      0.059310
paypal    0.051429
Name: is_fraud, dtype: float64

Fraud rates vary slightly by payment method, suggesting that some transaction types may carry higher risk than others.

## 3. Data Preparation


The data was cleaned and transformed to prepare it for modeling.

First, the `order_datetime` column was converted into a proper datetime format, and new features were created from it, including `order_hour` and `order_day`.

A new feature called `zip_mismatch` was created to indicate whether the billing and shipping ZIP codes are different, which may be a signal of fraudulent behavior.

Unnecessary columns such as IDs and unused text fields were removed to avoid noise in the model.

Categorical variables (such as payment method, device type, and shipping state) were converted into numeric format using one-hot encoding.

Finally, the dataset was split into features (`X`) and target variable (`y`), making it ready for machine learning models.

In [11]:
# Creating a working copy of the original data
df = orders.copy()

In [12]:
# Convert order_datetime to proper datetime format

df["order_datetime"] = pd.to_datetime(df["order_datetime"])

In [13]:
# Extract useful time features from the datetime

df["order_hour"] = df["order_datetime"].dt.hour      # hour of the day (0–23)
df["order_day"] = df["order_datetime"].dt.dayofweek  # day of week (0=Monday)

In [14]:
# Create a feature to detect if billing and shipping ZIP codes are different

df["zip_mismatch"] = (df["billing_zip"] != df["shipping_zip"]).astype(int)

In [15]:
# Remove columns that are not useful for prediction
# - IDs don’t help the model learn patterns
# - order_datetime already transformed into features
# - promo_code has many missing values and adds noise

df = df.drop(columns=[
    "order_id",
    "customer_id",
    "order_datetime",
    "promo_code"
])

In [16]:
# Convert categorical variables into numeric format using one-hot encoding

df = pd.get_dummies(df, columns=[
    "payment_method",
    "device_type",
    "shipping_state"
], drop_first=True)

In [17]:
# FIX: remove ip_country (it is still text and breaking the model)

if "ip_country" in df.columns:
    df = df.drop(columns=["ip_country"])

In [18]:
# Separate the dataset into features (X) and target variable (y)
# X = all input variables
# y = what we are trying to predict (is_fraud)

X = df.drop("is_fraud", axis=1)
y = df["is_fraud"]

In [19]:
# FINAL CHECK

df.head()
print("\nShape:", df.shape)


Shape: (5000, 34)


## 4. Modeling
Several classification models were trained to predict whether an order is fraudulent.

A baseline logistic regression model was used first to establish a simple starting point. Because the dataset is imbalanced, a second logistic regression model with class weighting was trained to better handle fraudulent cases.

Finally, a Random Forest model was trained as an ensemble method to capture more complex patterns in the data.

In [20]:
# Split data into training and testing sets
# Training = what the model learns from
# Testing = what we use to evaluate performance

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

In [21]:
# Train a baseline model (Logistic Regression)

from sklearn.linear_model import LogisticRegression

model = LogisticRegression(max_iter=5000, solver="liblinear")

model.fit(X_train, y_train)

,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary ` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :term:`mul

In [22]:
# Make predictions
y_pred = model.predict(X_test)

In [23]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

print("Accuracy:", accuracy_score(y_test, y_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

Accuracy: 0.926

Confusion Matrix:
[[926   5]
 [ 69   0]]

Classification Report:
              precision    recall  f1-score   support

           0       0.93      0.99      0.96       931
           1       0.00      0.00      0.00        69

    accuracy                           0.93      1000
   macro avg       0.47      0.50      0.48      1000
weighted avg       0.87      0.93      0.90      1000



In [25]:
# Train a logistic regression model that handles class imbalance

balanced_model = LogisticRegression(
    max_iter=5000,
    solver="liblinear",
    class_weight="balanced"
)

balanced_model.fit(X_train, y_train)

,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",'balanced'
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary ` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :ter

In [26]:
# Make predictions with the balanced model
y_pred_balanced = balanced_model.predict(X_test)

# Evaluate balanced model
print("Balanced Model Accuracy:", accuracy_score(y_test, y_pred_balanced))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_balanced))

print("\nClassification Report:")
print(classification_report(y_test, y_pred_balanced))

Balanced Model Accuracy: 0.75

Confusion Matrix:
[[709 222]
 [ 28  41]]

Classification Report:
              precision    recall  f1-score   support

           0       0.96      0.76      0.85       931
           1       0.16      0.59      0.25        69

    accuracy                           0.75      1000
   macro avg       0.56      0.68      0.55      1000
weighted avg       0.91      0.75      0.81      1000



In [27]:
from sklearn.ensemble import RandomForestClassifier

# Train Random Forest model
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)

rf_model.fit(X_train, y_train)

# Predictions
y_pred_rf = rf_model.predict(X_test)

# Evaluation
print("Random Forest Accuracy:", accuracy_score(y_test, y_pred_rf))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_rf))

print("\nClassification Report:")
print(classification_report(y_test, y_pred_rf))

Random Forest Accuracy: 0.93

Confusion Matrix:
[[930   1]
 [ 69   0]]

Classification Report:
              precision    recall  f1-score   support

           0       0.93      1.00      0.96       931
           1       0.00      0.00      0.00        69

    accuracy                           0.93      1000
   macro avg       0.47      0.50      0.48      1000
weighted avg       0.87      0.93      0.90      1000



The Random Forest model achieved high overall accuracy (0.93), but failed to correctly identify any fraudulent transactions (recall = 0.00 for class 1). This indicates that the model is heavily biased toward the majority class.

Despite being a more complex ensemble method, Random Forest did not outperform the balanced logistic regression model in detecting fraud. This highlights the importance of evaluating models using appropriate metrics for imbalanced datasets, rather than relying solely on accuracy.

Overall, the balanced logistic regression model provided the best performance for fraud detection due to its ability to capture the minority class.

## 5. Evaluation
The baseline logistic regression and Random Forest models achieved high accuracy but failed to detect fraudulent transactions (recall = 0.00 for class 1). This is due to class imbalance, where most transactions are non-fraud.

The balanced logistic regression model significantly improved fraud detection, achieving a recall of 0.59 for fraudulent transactions. Although its overall accuracy is lower, it is more effective for identifying fraud.

For this problem, recall for the fraud class is more important than overall accuracy. Therefore, the balanced logistic regression model is selected as the best model.

In [28]:
# Feature importance using Random Forest (simple feature selection idea)

import pandas as pd

importances = rf_model.feature_importances_
feature_names = X.columns

feature_importance_df = pd.DataFrame({
    "feature": feature_names,
    "importance": importances
}).sort_values(by="importance", ascending=False)

feature_importance_df.head(10)

,feature,importance
7,risk_score,0.153357
5,tax_amount,0.127252
3,order_subtotal,0.126461
6,order_total,0.126426
8,order_hour,0.079101
4,shipping_fee,0.072171
1,shipping_zip,0.055205
9,order_day,0.053445
0,billing_zip,0.046959
14,device_type_mobile,0.019403


The feature importance results show that variables such as risk_score, order_total, and tax_amount are among the most influential in predicting fraud.

This suggests that both transaction characteristics and risk-related indicators play a key role in identifying fraudulent orders.

## 6. Deployment

The final model (balanced logistic regression) was saved using joblib so it can be used in a production environment.

This saved model can be loaded by an application or API to make predictions on new transactions in real time.

In [29]:
import joblib
joblib.dump(balanced_model, "fraud_model.pkl")

['fraud_model.pkl']

In [30]:
# Create a dataframe with predictions

predictions_df = X_test.copy()
predictions_df["actual"] = y_test.values
predictions_df["predicted"] = y_pred_balanced

In [31]:
# Save predictions to JSON file

predictions_df.to_json("predictions.json", orient="records")